# comparison of calibrations across keyframes, as we've noticed they differ. 

In [1]:
# ============================================================
# Imports + parquet index
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2

# ZOD SDK
from zod.data_classes.info import Information
from zod.data_classes.frame import ZodFrame
from zod.constants import Camera, Lidar

# Parquet index
INDEX_PATH = Path("/home/edgelab/multimodal-MoE/outputs/index/ZODmoe_frames_with_xyxy_bboxes_and_solar_bins.parquet")
df = pd.read_parquet(INDEX_PATH)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Rows: 99999
Columns: ['frame_id', 'time', 'image_path', 'resized_image_path', 'orig_w', 'orig_h', 'new_w', 'new_h', 'sx', 'sy', 'scraped_weather', 'time_of_day', 'solar_angle_elevation', 'country_code', 'road_type', 'road_condition', 'ped_count_clear', 'ped_count_unclear', 'ped_occ_none', 'ped_occ_light', 'ped_occ_medium', 'ped_occ_heavy', 'ped_occ_veryheavy', 'ped_occ_missing', 'ped_occ_unknown', 'ped_uuid', 'ped_unclear_list', 'ped_occlusion_list', 'ped_points_xy_resized', 'ped_bin_4', 'ped_present', 'xyxy_bboxes', 'solar_context_bin']


In [3]:
# ============================================================
# Build ZodFrame from raw single-frame folder
# ============================================================

def make_zodframe_from_raw(frame_dir: Path) -> ZodFrame:
    """
    Build ZodFrame from raw single-frame directory layout.
    We need this to create ZodFrame objects from raw frame directories, 
    Since our data structure is not exactly the same as the SDK's.
    This fixes relative paths in info.json to be frame-local.
    """
    info_dict = json.loads((frame_dir / "info.json").read_text())
    fid = info_dict.get("id", frame_dir.name) #get frame_id from info.json dict
    prefix = f"single_frames/{fid}/" #

    def fix_path(p):
        if p is None:
            return None
        if p.startswith(prefix):
            return p[len(prefix):]
        if p.startswith("single_frames/"):
            return p.split("/", 2)[-1]
        return p

    # top-level paths
    for k in ["calibration_path", "ego_motion_path", "metadata_path", "oxts_path", "vehicle_data_path"]:
        if info_dict.get(k) is not None:
            info_dict[k] = fix_path(info_dict[k])

    # annotation paths
    for ann in info_dict.get("annotations", {}).values():
        if ann.get("filepath") is not None:
            ann["filepath"] = fix_path(ann["filepath"])

    # camera frame paths
    for arr in info_dict.get("camera_frames", {}).values():
        for x in arr:
            if x.get("filepath") is not None:
                x["filepath"] = fix_path(x["filepath"])

    # lidar frame paths
    for arr in info_dict.get("lidar_frames", {}).values():
        for x in arr:
            if x.get("filepath") is not None:
                x["filepath"] = fix_path(x["filepath"])

    info = Information.from_dict(info_dict)
    info.convert_paths_to_absolute(str(frame_dir))
    return ZodFrame(info)

In [4]:
# ============================================================
# Pick the two keyframes to compare
# ============================================================

FRAME_ID_A = "000000"
FRAME_ID_B = "069569"

print("Comparing frame_ids:", FRAME_ID_A, "vs", FRAME_ID_B)

Comparing frame_ids: 000000 vs 069569


In [5]:
# ============================================================
# Recover raw frame_dir from parquet
# ============================================================

def get_frame_dir_from_parquet(df: pd.DataFrame, frame_id: str) -> Path:
    """
    Given a frame_id, return the corresponding frame_dir from the parquet index.
    For example, if the parquet index has a row with frame_id = 000000,
    and image_path = /home/edgelab/multimodal-MoE/data/ZOD/single_frames/000000/camera_front_dnat/000000.jpg,
    then the frame_dir is /home/edgelab/multimodal-MoE/data/ZOD/single_frames/000000/.
    Function uses the image_path to find the frame_dir by looking at the parent directory of the image_path.
    """
    matches = df[df["frame_id"].astype(str) == str(frame_id)]
    if len(matches) == 0:
        raise ValueError(f"frame_id {frame_id} not found in parquet index")

    row = matches.iloc[0]
    image_path = Path(row["image_path"])      # original non-resized image path
    frame_dir = image_path.parent.parent      # .../<frame_id>/camera_front_dnat/<img>.jpg -> .../<frame_id>
    return frame_dir

frame_dir_a = get_frame_dir_from_parquet(df, FRAME_ID_A)
frame_dir_b = get_frame_dir_from_parquet(df, FRAME_ID_B)

print("frame_dir_a:", frame_dir_a)
print("frame_dir_b:", frame_dir_b)

frame_dir_a: /home/edgelab/zod_dino_data/train2017/000000
frame_dir_b: /home/edgelab/zod_dino_data/train2017/069569


In [6]:
# ============================================================
# Build the two ZodFrame objects
# ============================================================

zod_frame_a = make_zodframe_from_raw(frame_dir_a)
zod_frame_b = make_zodframe_from_raw(frame_dir_b)

calib_a = zod_frame_a.calibration
calib_b = zod_frame_b.calibration

# get front camera calibration objects (intrinsics, extrinsics, distortion)
cam_a = calib_a.cameras[Camera.FRONT]
cam_b = calib_b.cameras[Camera.FRONT]

# get LiDAR calibration objects (extrinsics)
lidar_a = calib_a.lidars[Lidar.VELODYNE]
lidar_b = calib_b.lidars[Lidar.VELODYNE]

print("zod_frame_a id:", zod_frame_a.info.id)
print("zod_frame_b id:", zod_frame_b.info.id)
print()
print("Front camera intrinsics shape A:", cam_a.intrinsics.shape)
print("Front camera intrinsics shape B:", cam_b.intrinsics.shape)

zod_frame_a id: 000000
zod_frame_b id: 069569

Front camera intrinsics shape A: (3, 4)
Front camera intrinsics shape B: (3, 4)


In [ ]:
# ============================================================
# Extract calibration pieces exactly how the SDK stores them
# - camera extrinsics and lidar extrinsics are Pose objects
# - the actual 4x4 matrices live in .transform
# ============================================================

# -------------------------
# Front camera calibration
# -------------------------
intr_a = np.array(cam_a.intrinsics)
intr_b = np.array(cam_b.intrinsics)

cam_ext_a = np.array(cam_a.extrinsics.transform)
cam_ext_b = np.array(cam_b.extrinsics.transform)

dist_a = np.array(cam_a.distortion)
dist_b = np.array(cam_b.distortion)

undist_a = np.array(cam_a.undistortion)
undist_b = np.array(cam_b.undistortion)

fov_a = np.array(cam_a.field_of_view)
fov_b = np.array(cam_b.field_of_view)

imgdim_a = np.array(cam_a.image_dimensions)
imgdim_b = np.array(cam_b.image_dimensions)

# -------------------------
# LiDAR calibration
# -------------------------
lidar_ext_a = np.array(lidar_a.extrinsics.transform)
lidar_ext_b = np.array(lidar_b.extrinsics.transform)

print("intrinsics shape:", intr_a.shape)
print("camera extrinsics shape:", cam_ext_a.shape)
print("lidar extrinsics shape:", lidar_ext_a.shape)
print("distortion shape:", dist_a.shape)
print("undistortion shape:", undist_a.shape)
print("field_of_view shape:", fov_a.shape)
print("image_dimensions shape:", imgdim_a.shape)

intrinsics shape: (3, 4)
camera extrinsics shape: (4, 4)
lidar extrinsics shape: (4, 4)
distortion shape: (4,)
undistortion shape: (4,)
field_of_view shape: (2,)
image_dimensions shape: (2,)


In [ ]:
# ============================================================
# Equality checks
# ============================================================

checks = {
    "intrinsics_equal": np.allclose(intr_a, intr_b),
    "camera_extrinsics_equal": np.allclose(cam_ext_a, cam_ext_b),
    "lidar_extrinsics_equal": np.allclose(lidar_ext_a, lidar_ext_b),
    "distortion_equal": np.allclose(dist_a, dist_b),
    "undistortion_equal": np.allclose(undist_a, undist_b),
    "field_of_view_equal": np.allclose(fov_a, fov_b),
    "image_dimensions_equal": np.allclose(imgdim_a, imgdim_b),
}

pd.DataFrame({
    "component": list(checks.keys()),
    "equal": list(checks.values())
})

,component,equal
0,intrinsics_equal,False
1,camera_extrinsics_equal,False
2,lidar_extrinsics_equal,False
3,distortion_equal,False
4,undistortion_equal,False
5,field_of_view_equal,False
6,image_dimensions_equal,True


In [ ]:
# ============================================================
# Compact scalar summary
# ============================================================

summary_df = pd.DataFrame({
    "parameter": ["fx", "fy", "cx", "cy", "fov_horizontal_deg", "fov_vertical_deg", "img_w", "img_h"],
    FRAME_ID_A: [
        intr_a[0, 0],
        intr_a[1, 1],
        intr_a[0, 2],
        intr_a[1, 2],
        fov_a[0],
        fov_a[1],
        imgdim_a[0],
        imgdim_a[1],
    ],
    FRAME_ID_B: [
        intr_b[0, 0],
        intr_b[1, 1],
        intr_b[0, 2],
        intr_b[1, 2],
        fov_b[0],
        fov_b[1],
        imgdim_b[0],
        imgdim_b[1],
    ],
})

summary_df["delta"] = summary_df[FRAME_ID_B] - summary_df[FRAME_ID_A]
summary_df

,parameter,000000,069569,delta
0,fx,1866.254650,1857.349653,-8.904997
1,fy,1866.254650,1857.349653,-8.904997
2,cx,1919.254029,1917.924624,-1.329406
3,cy,1070.538147,1115.149285,44.611138
4,fov_horizontal_deg,120.031046,121.160484,1.129437
5,fov_vertical_deg,66.953625,67.287317,0.333693
6,img_w,3848.000000,3848.000000,0.000000
7,img_h,2168.000000,2168.000000,0.000000


In [23]:
# ============================================================
# Cell 9: Distortion / undistortion comparison
# ============================================================

dist_df = pd.DataFrame({
    "coef": ["k1", "k2", "k3", "k4"],
    f"{FRAME_ID_A}_distortion": dist_a,
    f"{FRAME_ID_B}_distortion": dist_b,
    f"{FRAME_ID_A}_undistortion": undist_a,
    f"{FRAME_ID_B}_undistortion": undist_b,
})

dist_df["distortion_delta"] = dist_df[f"{FRAME_ID_B}_distortion"] - dist_df[f"{FRAME_ID_A}_distortion"]
dist_df["undistortion_delta"] = dist_df[f"{FRAME_ID_B}_undistortion"] - dist_df[f"{FRAME_ID_A}_undistortion"]

dist_df

,coef,000000_distortion,069569_distortion,000000_undistortion,069569_undistortion,distortion_delta,undistortion_delta
0,k1,-0.021352,-0.005930,0.021528,0.005142,0.015422,-0.016386
1,k2,0.017424,-0.053658,-0.017535,0.059334,-0.071082,0.076869
2,k3,-0.018277,0.065915,0.019734,-0.073066,0.084192,-0.092800
3,k4,0.007463,-0.024769,-0.008409,0.027989,-0.032232,0.036398


In [24]:
# ============================================================
# Cell 10: Actual matrices
# ============================================================

print("============================================================")
print(f"INTRINSICS: {FRAME_ID_A}")
print("============================================================")
print(intr_a)
print()

print("============================================================")
print(f"INTRINSICS: {FRAME_ID_B}")
print("============================================================")
print(intr_b)
print()

print("============================================================")
print(f"CAMERA EXTRINSICS: {FRAME_ID_A}")
print("============================================================")
print(cam_ext_a)
print()

print("============================================================")
print(f"CAMERA EXTRINSICS: {FRAME_ID_B}")
print("============================================================")
print(cam_ext_b)
print()

print("============================================================")
print(f"LIDAR EXTRINSICS: {FRAME_ID_A}")
print("============================================================")
print(lidar_ext_a)
print()

print("============================================================")
print(f"LIDAR EXTRINSICS: {FRAME_ID_B}")
print("============================================================")
print(lidar_ext_b)

INTRINSICS: 000000
[[1.86625465e+03 0.00000000e+00 1.91925403e+03 0.00000000e+00]
 [0.00000000e+00 1.86625465e+03 1.07053815e+03 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00 0.00000000e+00]]

INTRINSICS: 069569
[[1.85734965e+03 0.00000000e+00 1.91792462e+03 0.00000000e+00]
 [0.00000000e+00 1.85734965e+03 1.11514928e+03 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00 0.00000000e+00]]

CAMERA EXTRINSICS: 000000
[[ 3.14911818e-03  7.39364209e-03  9.99967708e-01  2.02307369e+00]
 [-9.99901564e-01  1.36957403e-02  3.04764521e-03 -1.72912025e-03]
 [-1.36727648e-02 -9.99878873e-01  7.43604379e-03  1.12993067e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]

CAMERA EXTRINSICS: 069569
[[-1.96880384e-02 -5.45243113e-03  9.99791304e-01  2.01530662e+00]
 [-9.99698707e-01 -1.45539035e-02 -1.97655857e-02  4.90213071e-04]
 [ 1.46586366e-02 -9.99879220e-01 -5.16425054e-03  1.12506001e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.0000

In [ ]:
# ============================================================
# Matrix deltas
# ============================================================

intr_delta = intr_b - intr_a
cam_ext_delta = cam_ext_b - cam_ext_a
lidar_ext_delta = lidar_ext_b - lidar_ext_a

print("============================================================")
print(f"INTRINSICS DELTA ({FRAME_ID_B} - {FRAME_ID_A})")
print("============================================================")
print(intr_delta)
print()

print("============================================================")
print(f"CAMERA EXTRINSICS DELTA ({FRAME_ID_B} - {FRAME_ID_A})")
print("============================================================")
print(cam_ext_delta)
print()

print("============================================================")
print(f"LIDAR EXTRINSICS DELTA ({FRAME_ID_B} - {FRAME_ID_A})")
print("============================================================")
print(lidar_ext_delta)

INTRINSICS DELTA (069569 - 000000)
[[-8.90499662  0.         -1.32940569  0.        ]
 [ 0.         -8.90499662 44.61113806  0.        ]
 [ 0.          0.          0.          0.        ]]

CAMERA EXTRINSICS DELTA (069569 - 000000)
[[-2.28371566e-02 -1.28460732e-02 -1.76403742e-04 -7.76706892e-03]
 [ 2.02857064e-04 -2.82496437e-02 -2.28132309e-02  2.21933332e-03]
 [ 2.83314014e-02 -3.47108997e-07 -1.26002943e-02 -4.87065788e-03]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]]

LIDAR EXTRINSICS DELTA (069569 - 000000)
[[ 6.15557127e-03  1.82032204e-04 -6.18595945e-04  1.01445393e-03]
 [-1.69161553e-04  6.15598412e-03 -2.04646371e-03 -1.08734436e-02]
 [-2.07580733e-03  5.93104451e-04 -1.05990679e-05 -3.27250859e-03]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]]
